# Managed resource lifecycle tests

This notebook checks synchronous and asynchronous cleanup paths for `boti.core.managed_resource`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [2]:
from boti.core import ManagedResource

class SimpleResource(ManagedResource):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.cleaned_up_sync = False
        self.cleaned_up_async = False

    def _cleanup(self):
        self.cleaned_up_sync = True

    async def _acleanup(self):
        self.cleaned_up_async = True

class SyncOnlyResource(ManagedResource):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.cleaned_up = False

    def _cleanup(self):
        self.cleaned_up = True

sync_resource = SimpleResource()
with sync_resource as resource:
    assert not resource.closed
assert sync_resource.closed
assert sync_resource.cleaned_up_sync

print("Synchronous lifecycle passed.")

Synchronous lifecycle passed.


In [3]:
async def exercise_async_paths():
    async_resource = SimpleResource()
    async with async_resource as resource:
        assert not resource.closed
    assert async_resource.closed
    assert async_resource.cleaned_up_async

    sync_only = SyncOnlyResource()
    await sync_only.aclose()
    assert sync_only.closed
    assert sync_only.cleaned_up

await exercise_async_paths()
print("Asynchronous lifecycle passed.")

Asynchronous lifecycle passed.
